# Reference-profile slope workflow

This is the clean, example-driven explanation and QA notebook for the promoted slope workflow. It is read-only: it reconstructs and inspects one selected base network, but never overwrites hydraulic contexts, equivalent-resistance products, or RAPID products.

The result is a positive longitudinal water-surface profile on the frozen single-centerline reference. That profile supplies trusted outer boundary heads; realised braided-network slopes are then assigned with simple allocation or constrained domain solves.

## Workflow at a glance

1. Freeze one realised base network and its single-centerline reference.
2. Identify the two network endpoint anchors and construct the endpoint-bounded SWORD reach corridor.
3. Retain all screened SWORD nodes inside that corridor and all valid multi-pass SWOT RiverSP observations at those nodes.
4. Estimate a shared, smooth, strictly downstream-decreasing WSE profile from all passes.
5. Interpolate that profile to trusted network reference anchors.
6. Allocate slopes to simple units, solve compound domains, and retain direct profile slopes for connectors.
7. Re-use fixed external boundary heads to solve changed child domains after collapse.
8. audit positivity, local consistency, and path head-loss closure before promotion.

The notebook deliberately separates **measurement/profile fitting** from **network slope allocation**. The first uses SWORD and SWOT; the second uses the realised network topology and geometric link lengths.

## 0. Theory and definitions

Let (x) be distance to outlet along the frozen single centerline, so (x) increases upstream. Let (z_{ij}) be a valid RiverSP water-surface observation at SWORD node (i), acquired during pass (j).

We fit

$[z_{ij}=g(x_i)+a_j+\epsilon_{ij}]$,

where:

- (g(x)) is the shared spatial WSE profile;
- (a_j) is an acquisition-specific stage offset;
- $(\sum_j a_j=0)$, so the fitted profile remains on the common observed datum;
- $(\epsilon_{ij})$ is remaining measurement and local hydraulic variation.

On the retained SWORD-node grid, the fit minimises weighted data misfit plus a natural-spline curvature penalty:

$\min_{g,a}\ \sum_{ij}w_{ij}[z_{ij}-g(x_i)-a_j]^2
+\lambda\,g^\mathsf{T}Pg$,

subject to a small positive longitudinal slope:

$[g(x_{i+1})-g(x_i)\ge S_{\min}(x_{i+1}-x_i),
\qquad S_{\min}=0.01\,S_{\rm global}]$.

The smoothing multiplier is selected by whole-acquisition cross-validation with the one-standard-error rule. When the smooth tail is demonstrably converged, the audited minimum-curvature continuation is retained. The profile therefore uses all screened multi-pass information while remaining physically monotone and positive-gradient.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "network_variants").is_dir() and (candidate / "RAPID").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / "RAPID" / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "RAPID" / "src"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rapid_tools.reference_profile import (
    ReferenceProfileConfig,
    fit_reference_profile,
    select_reference_profile,
)
from rapid_tools.reference_profile_inputs import build_reference_profile_inputs
from rapid_tools.connector_support_windows import audit_connector_physical_support
from network_variants.hydraulic_context import _load_reference_section_support
from network_variants.reference_profile_candidate import solve_boundary_conditioned_domain
from network_variants.sword_matching import load_sword_node_reference

print(f"Repository: {REPO_ROOT}")

## 1. Select one promoted base network

Choose a base independent experiment. The notebook never uses a collapsed mask as the reference geometry: every slope profile is built from the frozen, realised base network and its established single-centerline reference.

The default example can be changed freely. For the direction-corrected Example 15, use the promoted `15_indep_direction130__anchored` root.

In [ ]:
EXAMPLE_ID = "11"
EXPERIMENT_NAME = f"{EXAMPLE_ID}_indep__anchored"
STATE_ID = "S000_base"

OUTPUT_ROOT = Path("/Volumes/PhD/river_hierarchy/network_variants/output")
SWORD_NODE_SOURCE = Path("/Volumes/PhD/SWORD/v17c")
RIVERSP_AUDIT_OBSERVATIONS = REPO_ROOT / "data_validation" / "swot_sword_17c" / "target_observations.parquet"

# The promoted method uses equal valid-observation weight, a 1% global-slope
# floor, and acquisition-level cross-validation. These are inspection settings;
# this notebook writes no artifacts.
PROFILE_CONFIG = ReferenceProfileConfig(
    minimum_slope_fraction=0.01,
    weight_mode="equal",
    cv_folds=3,
)

EXPERIMENT_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
STATE_DIR = EXPERIMENT_DIR / "states" / STATE_ID
VARIANT_DIR = STATE_DIR / "variant"
REFERENCE_SECTION_SUMMARY = EXPERIMENT_DIR / "reference_section" / "reference_section_summary.json"

if not STATE_DIR.exists():
    raise FileNotFoundError(f"Missing promoted base state: {STATE_DIR}")
if not RIVERSP_AUDIT_OBSERVATIONS.exists():
    raise FileNotFoundError(f"Missing filtered RiverSP audit observations: {RIVERSP_AUDIT_OBSERVATIONS}")

print(f"Example: {EXAMPLE_ID}")
print(f"Experiment: {EXPERIMENT_DIR}")

## 2. Freeze the realised reference geometry and locate the network anchors

The two outer anchors are the unique source and sink of the realised directed base network. They define the portion of the single-centerline reference for which a WSE profile is needed.

They are not selected from arbitrary nearest SWORD nodes. We first identify the realised network endpoints, then project them to the reviewed SWORD mainstem corridor.

In [ ]:
def one_path(directory: Path, pattern: str) -> Path:
    matches = sorted(p for p in directory.glob(pattern) if not p.name.startswith("._"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one {pattern!r} in {directory}; found {matches}")
    return matches[0]


BASE_LINKS = gpd.read_file(one_path(VARIANT_DIR / "directed", "*directed_links.gpkg"))
BASE_NODES = gpd.read_file(one_path(VARIANT_DIR / "directed", "*directed_nodes.gpkg"))
SUPPORT = _load_reference_section_support(str(REFERENCE_SECTION_SUMMARY))
REFERENCE_LINE = SUPPORT["line"]
CENTERLINE_CRS = SUPPORT["centerline_crs"]

in_degree = BASE_LINKS.groupby("id_ds_node").size().rename("in_degree")
out_degree = BASE_LINKS.groupby("id_us_node").size().rename("out_degree")
node_degree = pd.concat([in_degree, out_degree], axis=1).fillna(0)
network_nodes = BASE_NODES.to_crs(CENTERLINE_CRS).join(node_degree, on="id_node").fillna({"in_degree": 0, "out_degree": 0})
network_anchors = network_nodes.loc[
    network_nodes["in_degree"].eq(0) | network_nodes["out_degree"].eq(0)
].copy()
network_anchors["anchor_role"] = np.where(network_anchors["in_degree"].eq(0), "network_source", "network_sink")

if len(network_anchors) != 2:
    raise RuntimeError(f"Expected exactly two realised endpoint anchors; found {len(network_anchors)}")

display(network_anchors[["id_node", "anchor_role", "in_degree", "out_degree", "geometry"]])

## 3. Construct the endpoint-bounded SWORD corridor

The corridor procedure is fixed and reproducible:

1. Project each realised network endpoint to its closest reviewed SWORD mainstem reach.
2. Follow the reviewed SWORD downstream main path between those two reaches.
3. Extend a corridor end by one adjoining reach only when the projected endpoint lies within the 1 km endpoint buffer of that reach end.
4. Retain all SWORD nodes belonging to those reaches, but remove nodes beyond the two projected network endpoints.

Thus the retained reaches provide small geometric tails for review, while the retained node population represents exactly the network-bounded longitudinal corridor.

In [ ]:
profile_inputs = build_reference_profile_inputs(
    EXPERIMENT_DIR,
    sword_node_source=SWORD_NODE_SOURCE,
    raw_observations_path=RIVERSP_AUDIT_OBSERVATIONS,
    max_centerline_offset_m=350.0,
    endpoint_buffer_m=1_000.0,
)

corridor_metadata = pd.DataFrame(profile_inputs.metadata["endpoint_projection"])
display(corridor_metadata)
print("Corridor reach IDs:", profile_inputs.metadata["corridor_reach_ids"])
print("Retained profile nodes:", profile_inputs.metadata["n_static_profile_nodes"])
print("Filtered RiverSP observations:", profile_inputs.metadata["n_raw_observations"])
print("Independent acquisitions:", profile_inputs.metadata["n_raw_acquisitions"])

In [ ]:
# Spatial QA map. Each layer is independently selectable in the Plotly legend.
region = str(profile_inputs.metadata["sword_region"]).upper()
reach_ids = profile_inputs.metadata["corridor_reach_ids"]
reach_path = SWORD_NODE_SOURCE / f"sword_{region}_v17c_reaches.parquet"
corridor_reaches = gpd.read_parquet(reach_path, columns=["reach_id", "geometry"]).to_crs(CENTERLINE_CRS)
corridor_reaches = corridor_reaches.loc[corridor_reaches["reach_id"].isin(reach_ids)].copy()
corridor_nodes = load_sword_node_reference(
    SWORD_NODE_SOURCE, sword_wse_field="wse_obs_p50", regions=[region.lower()], reach_ids=reach_ids
).to_crs(CENTERLINE_CRS)

fig = go.Figure()
for frame, name, color, width in [
    (BASE_LINKS.to_crs(CENTERLINE_CRS), "realised base network", "#1f77b4", 1.5),
    (corridor_reaches, "SWORD corridor reaches", "#7f7f7f", 2.0),
]:
    for geom in frame.geometry:
        coords = np.asarray(geom.coords)
        fig.add_scatter(x=coords[:, 0], y=coords[:, 1], mode="lines", name=name, legendgroup=name,
                        showlegend=(len(fig.data) == 0 or name not in [trace.name for trace in fig.data]),
                        line={"color": color, "width": width})
fig.add_scatter(x=network_anchors.geometry.x, y=network_anchors.geometry.y, mode="markers+text",
                text=network_anchors["anchor_role"], textposition="top center", name="network anchors",
                marker={"color": "#d62728", "size": 10})
fig.add_scatter(x=corridor_nodes.geometry.x, y=corridor_nodes.geometry.y, mode="markers", name="retained SWORD nodes",
                marker={"color": "#2ca02c", "size": 4})
fig.update_layout(title="Frozen realised network and endpoint-bounded SWORD corridor", yaxis={"scaleanchor": "x", "scaleratio": 1})
fig.show()

## 4. Build the multi-pass SWOT/RiverSP observation set

Every retained SWORD node contributes all valid RiverSP observations passing the fixed quality screen. The static SWORD P50 value is retained as an initial/reference profile, but it is not treated as the only measurement.

The plot below shows raw node P50 and the empirical P10–P90 envelope from the screened multi-pass observations. The envelope is descriptive evidence, not a fitted confidence interval.

In [ ]:
static_profile = profile_inputs.static_profile.copy()
raw_observations = profile_inputs.observations.copy()

observation_envelope = raw_observations.groupby("reference_section_dist_out_m", as_index=False).agg(
    n_observations=("wse_m", "size"),
    raw_p10_m=("wse_m", lambda values: values.quantile(0.10)),
    raw_p50_m=("wse_m", "median"),
    raw_p90_m=("wse_m", lambda values: values.quantile(0.90)),
).sort_values("reference_section_dist_out_m")

display(observation_envelope.head())

fig = go.Figure()
fig.add_scatter(x=observation_envelope["reference_section_dist_out_m"] / 1_000, y=observation_envelope["raw_p90_m"],
                mode="lines", line={"width": 0}, showlegend=False, hoverinfo="skip")
fig.add_scatter(x=observation_envelope["reference_section_dist_out_m"] / 1_000, y=observation_envelope["raw_p10_m"],
                mode="lines", fill="tonexty", fillcolor="rgba(120,120,120,0.22)", line={"width": 0},
                name="screened RiverSP P10–P90 envelope")
fig.add_scatter(x=observation_envelope["reference_section_dist_out_m"] / 1_000, y=observation_envelope["raw_p50_m"],
                mode="markers", name="screened RiverSP P50", marker={"size": 4, "color": "#555555"})
fig.add_scatter(x=static_profile["reference_section_dist_out_m"] / 1_000, y=static_profile["static_wse_m"],
                mode="lines", name="static SWORD P50 reference", line={"color": "#9467bd"})
fig.update_layout(title="Screened raw RiverSP observations", xaxis_title="Distance to outlet [km]", yaxis_title="WSE [m]")
fig.show()

## 5. Fit and select the monotone multi-pass profile

The spline is represented on **every retained SWORD-node coordinate**; there is no arbitrary knot-spacing parameter. The only smoothness choice is (lambda), selected using cross-validation that withholds entire acquisitions rather than individual observations.

The optimisation alternates between:

- estimating the monotone, positive-gradient spatial profile (g(x)); and
- estimating pass offsets (a_j), re-centred after every update so their mean remains zero.

Run this cell only when re-performing a fit. It is computationally substantial for long examples. The selected profile is diagnostic unless it is separately written, reviewed, and promoted through the production command path.

In [ ]:
# Refit the selected example from raw multi-pass observations.
# This cell is intentionally explicit: no profile artifact is written here.
fit, cv_scores, selection = select_reference_profile(
    static_profile,
    raw_observations,
    endpoint_distances_m=profile_inputs.endpoint_distances_m,
    config=PROFILE_CONFIG,
)

print(json.dumps(selection, indent=2))
display(pd.DataFrame([fit.summary]))

fig = go.Figure()
fig.add_scatter(x=cv_scores["lambda_value"], y=cv_scores["cv_mean_mse_m2"], mode="lines+markers", name="CV mean MSE")
fig.add_scatter(x=cv_scores["lambda_value"], y=cv_scores["cv_mean_mse_m2"] + cv_scores["cv_se_mse_m2"],
                mode="lines", line={"width": 0}, showlegend=False, hoverinfo="skip")
fig.add_scatter(x=cv_scores["lambda_value"], y=cv_scores["cv_mean_mse_m2"] - cv_scores["cv_se_mse_m2"],
                mode="lines", fill="tonexty", line={"width": 0}, name="one standard error")
fig.add_vline(x=selection["selected_lambda"], line_dash="dash", line_color="#d62728", annotation_text="selected λ")
fig.update_xaxes(type="log", title="Spline curvature penalty λ")
fig.update_yaxes(title="Whole-acquisition cross-validation MSE [m²]")
fig.update_layout(title="Smoothing selection")
fig.show()

## 6. Inspect the accepted longitudinal profile

A valid final profile must satisfy all of the following:

- positive global head loss between the realised endpoint anchors;
- a strictly positive interval slope, with the small floor active only where necessary;
- reasonable agreement with the multi-pass observation envelope and pass-level traces;
- zero-mean acquisition offsets;
- no unexplained sharp feature that is unsupported by the screened observations.

The profile is an external hydraulic reference. It is not a claim that every braided link lies exactly on the single-centerline water surface.

In [ ]:
profile = fit.profile.copy()
profile_observations = fit.observations.copy()

fig = go.Figure()
for acquisition_id, group in profile_observations.groupby("acquisition_id", sort=False):
    fig.add_scatter(
        x=group["reference_section_dist_out_m"] / 1_000,
        y=group["wse_m"],
        mode="lines+markers", name=f"RiverSP pass {acquisition_id}", visible="legendonly",
        line={"width": 1}, marker={"size": 3}, opacity=0.35,
    )
fig.add_scatter(x=observation_envelope["reference_section_dist_out_m"] / 1_000, y=observation_envelope["raw_p90_m"],
                mode="lines", line={"width": 0}, showlegend=False, hoverinfo="skip")
fig.add_scatter(x=observation_envelope["reference_section_dist_out_m"] / 1_000, y=observation_envelope["raw_p10_m"],
                mode="lines", fill="tonexty", fillcolor="rgba(120,120,120,0.18)", line={"width": 0},
                name="screened P10–P90 envelope")
fig.add_scatter(x=profile["reference_section_dist_out_m"] / 1_000, y=profile["fitted_wse_m"],
                mode="lines", name="selected monotone multi-pass profile", line={"color": "#d62728", "width": 3})
fig.add_scatter(x=static_profile["reference_section_dist_out_m"] / 1_000, y=static_profile["static_wse_m"],
                mode="lines", name="static SWORD P50 reference", line={"color": "#9467bd", "dash": "dash"})
fig.update_layout(title="Selected reference WSE profile", xaxis_title="Distance to outlet [km]", yaxis_title="WSE [m]")
fig.show()

intervals = profile.dropna(subset=["interval_slope_m_per_m"])
print(f"Global profile slope: {fit.summary['fitted_global_slope_m_per_m']:.6g} m/m")
print(f"Minimum profile slope: {intervals['interval_slope_m_per_m'].min():.6g} m/m")
print(f"Floor-active intervals: {fit.summary['n_floor_active_intervals']}")
display(fit.acquisition_offsets.sort_values("stage_offset_m"))

## 7. Transfer profile heads to trusted network anchors

The fitted profile is interpolated only to realised network nodes that are valid reference anchors: outer network endpoints and the established boundaries of simple or compound domains.

For an anchored domain with outer heads (H_\mathrm{up}) and (H_\mathrm{down}), the profile fixes its total available head loss:

\[
\Delta H_\mathrm{domain}=H_\mathrm{up}-H_\mathrm{down}>0.
\]

This is the invariant boundary condition used in all later network allocation and domain solves.

In [ ]:
# The promoted node hydraulic-context table is the authoritative record of
# reference stations and profile heads used downstream.
node_context_path = STATE_DIR / "hydraulic_context" / "node_hydraulic_context.csv"
if not node_context_path.exists():
    raise FileNotFoundError(f"Missing promoted node hydraulic context: {node_context_path}")
node_context = pd.read_csv(node_context_path)

anchor_fields = [field for field in [
    "id_node", "reference_section_dist_out_m", "rapid_wse_for_slope",
    "sword_wse", "slope_profile_mode", "slope_source_method",
] if field in node_context.columns]
anchor_context = node_context.loc[node_context["id_node"].isin(network_anchors["id_node"])].copy()
display(anchor_context[anchor_fields])

## 8. Allocate simple-unit slopes

A simple unit is bounded by two trusted profile-anchor heads. Its reference slope is:

\[
S_{\rm ref}=\frac{H_\mathrm{up}-H_\mathrm{down}}{L_\mathrm{reference}}.
\]

The realised links receive positive drops over their **actual realised geometric lengths**, while preserving the fixed boundary head loss. Thus a topology change may alter realised path length and individual slopes without changing the external profile boundary condition.

The linked hydraulic-context table below is the promoted, downstream-consumed record; it should be used to inspect an individual simple unit.

In [ ]:
link_context_path = STATE_DIR / "hydraulic_context" / "link_hydraulic_context.csv"
if not link_context_path.exists():
    raise FileNotFoundError(f"Missing promoted link hydraulic context: {link_context_path}")
link_context = pd.read_csv(link_context_path)

SIMPLE_UNIT_ID = None  # e.g. 4; set an integer to inspect a specific simple unit.
if SIMPLE_UNIT_ID is None:
    print("Set SIMPLE_UNIT_ID to inspect one simple unit.")
else:
    simple_links = link_context.loc[link_context["unit_ids"].astype(str).str.fullmatch(str(SIMPLE_UNIT_ID), na=False)].copy()
    fields = [field for field in [
        "id_link", "id_us_node", "id_ds_node", "len", "len_adj", "slope_use",
        "slope_source_method", "slope_reason", "rapid_wse_us", "rapid_wse_ds",
    ] if field in simple_links.columns]
    display(simple_links[fields].sort_values("id_link"))

## 9. Solve base compound domains

A compound bubble has fixed profile-derived outer heads but multiple internal paths. The internal node heads are solved so every realised link remains positive-gradient and the network stays close to its local reference slope:

\[
\min_H\sum_{e\in\mathrm{bubble}}
\left[
\frac{(H_{u,e}-H_{d,e})/L_e-S_\mathrm{ref}}
{S_\mathrm{ref}}
\right]^2.
\]

Subject to:

- fixed outer heads;
- every realised directed link has slope at least (0.01S_\mathrm{ref});
- every source-to-sink path closes on the same prescribed boundary head loss.

Each link has equal influence in this objective. This avoids making short links a cheap place to concentrate otherwise implausible head loss.

In [ ]:
# The production solver is exposed here for a selected, already identified domain.
# Populate SELECTED_DOMAIN_LINKS and the four physical boundary values from the
# promoted hierarchy/context audits before executing this example solve.
SELECTED_COMPOUND_BUBBLE_ID = None

if SELECTED_COMPOUND_BUBBLE_ID is None:
    print("Set SELECTED_COMPOUND_BUBBLE_ID and derive its selected domain inputs to inspect one compound solve.")
else:
    raise NotImplementedError(
        "Use solve_boundary_conditioned_domain(selected_links, source=..., sink=..., "
        "source_head_m=..., sink_head_m=..., reference_distance_m=...) after loading "
        "the bubble's promoted boundary record. This deliberate guard prevents an "
        "ambiguous implicit domain selection."
    )

## 10. Screen profile-direct connector sections

Not every realised link belongs uniquely to a simple unit or compound bubble. These connector sections retain their direct frozen-profile slope unless a separately documented profile-level intervention is approved.

Their physical plausibility is audited against the fitted profile immediately upstream and downstream. The audit uses 1, 2, and 5 km support windows and a factor-five comparison band. A flagged result is an investigation signal, not permission for a silent per-link repair.

For a connector with slope (S_c), the support target is the geometric median of valid neighbouring support slopes. The factor diagnostic is:

\[
F_c=\max(S_c/S_\mathrm{support},\ S_\mathrm{support}/S_c).
\]

Only an explicit, recorded profile-level bridge may modify a weakly supported interval; this preserves continuity for every downstream consumer.

In [ ]:
# Example diagnostic call after resolving the connector's two endpoint stations.
# The helper is read-only and returns an audit dictionary.
CONNECTOR_EXAMPLE = None  # (upstream_x_m, downstream_x_m, connector_length_m, connector_slope_m_per_m)
if CONNECTOR_EXAMPLE is None:
    print("Set CONNECTOR_EXAMPLE to audit one profile-direct connector.")
else:
    upstream_x_m, downstream_x_m, connector_length_m, connector_slope = CONNECTOR_EXAMPLE
    connector_audit = audit_connector_physical_support(
        profile.rename(columns={"fitted_wse_m": "joint_monotone_wse_m"}),
        upstream_x_m=upstream_x_m,
        downstream_x_m=downstream_x_m,
        connector_length_m=connector_length_m,
        connector_slope_m_per_m=connector_slope,
    )
    display(pd.DataFrame([connector_audit]))

## 11. Solve collapsed child domains

A collapsed child does **not** fit a new external SWORD/SWOT profile. It inherits the trusted, promoted external profile and retains the relevant outer boundary heads. Only the altered internal realised network is re-solved.

For a changed compound domain, the same constrained equal-per-link relative-slope objective is used with the child’s realised links and lengths. This ensures that base and child states are compared under the same external hydraulic boundary conditions.

For unchanged links, inherited hydraulic context is retained. For new/changed links, the provenance table records whether the slope came from direct profile allocation, a base compound solve, a connector audit, or a child-domain solve.

## 12. Final validation and promotion contract

Before a profile/context is promoted, verify:

1. **Corridor integrity:** exactly one endpoint-bounded connected SWORD corridor; correct endpoint projections; no unwanted SWORD nodes outside the corridor.
2. **Observation integrity:** stated RiverSP quality filter, pass coverage, stage-offset centring, and inspection of unusual features in individual acquisitions.
3. **Profile validity:** positive global head loss, strictly positive interval slopes, limited and audited floor activity, and no unexplained deviation from the screened observation envelope.
4. **Network allocation:** every simple/compound/child domain has finite positive slopes and a recorded source method.
5. **Head-loss closure:** every directed source-to-sink path through a solved domain reproduces its prescribed profile boundary drop within tolerance.
6. **Outlier review:** local/global slope diagnostics, connector factor-five screen, and explicit manual-profile adjustments only where documented.

The promoted outputs consumed by equivalent resistance and RAPID are the reference-profile artifact, node hydraulic context, link hydraulic context, domain/path audits, and provenance fields. Neither downstream workflow refits this profile.